# 02 UCDP

**Owner:** Nicolas

**What this notebook does:** turns the UCDP conflict list into one row per country-year and a table of conflict periods for the timeline.

**Reads:** `UcdpPrioConflict_v26_1.xlsx`, `crosswalk_gw.csv`

**Writes:** `ucdp_cy.csv`, `ucdp_spans.csv`

This is a main notebook. It must run from top to bottom with Kernel → Restart and Run All.
Write simple code in small steps: one operation per line, print the result after each step. Your sandbox section for this notebook: **sandbox_nicolas, Part B**. Paste only the working lines, one step per cell, with a short note above each.

## Setup

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

RAW = "../data/raw/"
CLEAN = "../data/clean/"
pd.set_option("display.max_columns", 50)

## B1 Load and inspect

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


ucdp_raw = pd.read_excel(RAW + "UcdpPrioConflict_v26_1.xlsx")
print(ucdp_raw.shape)

print(ucdp_raw[["conflict_id", "year", "gwno_loc", "intensity_level"]].head(10))

has_comma = ucdp_raw["gwno_loc"].astype(str).str.contains(",")
print(has_comma.sum())
print(ucdp_raw[has_comma].head())

relevantrow = ucdp_raw["year"] >= 1950
ucdp_raw = ucdp_raw[relevantrow].copy()
print(ucdp_raw.shape)

## B2 One row per country per conflict-year

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here
rows = []

for conflict_id, year, gwno_loc, intensity in zip(ucdp_raw["conflict_id"], ucdp_raw["year"], ucdp_raw["gwno_loc"], ucdp_raw["intensity_level"]):
    for code_text in str(gwno_loc).split(","):
        gw_code = int(code_text.strip())
        rows.append({"conflict_id": conflict_id, "year": year, "gw_code": gw_code, "intensity_level": intensity})

ucdp_long = pd.DataFrame(rows)
print(ucdp_long.head())

print(len(ucdp_raw))
print(len(ucdp_long))
print(ucdp_raw["gwno_loc"].astype(str).str.count(",").sum())

## B3 Add iso3

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


crosswalk_gw = pd.read_csv(CLEAN + "crosswalk_gw.csv")
print(crosswalk_gw.head())

n_before = len(ucdp_long)
ucdp_long = pd.merge(ucdp_long, crosswalk_gw, on="gw_code", how="left", validate="many_to_one", indicator=True)
print(n_before)
print(len(ucdp_long))
print(ucdp_long["_merge"].value_counts())

ucdp_long = ucdp_long.drop(columns="_merge")
print(ucdp_long.columns)

## B4 Country-year table

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here

ucdp_cy = ucdp_long.groupby(["iso3", "year"], as_index=False).agg(n_conflicts=("conflict_id", "nunique"), max_intensity=("intensity_level", "max"))
ucdp_cy["conflict_active"] = 1
print(ucdp_cy[(ucdp_cy["iso3"] == "UKR") & (ucdp_cy["year"] == 2015)])

## B5 Conflict periods and save

Paste your working code from the sandbox. Add one sentence above it that says what it does.

In [ ]:
# your code here


ucdp_cy = ucdp_cy.sort_values(["iso3", "year"]).copy()

periods = []
for code, rows in ucdp_cy.groupby("iso3"):
    prev_year = None
    for year, intensity in zip(rows["year"], rows["max_intensity"]):
        if prev_year is None:
            start = year
            highest = intensity
        elif year != prev_year + 1:
            periods.append({"iso3": code, "start": start, "end": prev_year, "max_intensity": highest})
            start = year
            highest = intensity
        else:
            highest = max(highest, intensity)
        prev_year = year
    periods.append({"iso3": code, "start": start, "end": prev_year, "max_intensity": highest})

ucdp_spans = pd.DataFrame(periods)
print(ucdp_spans.head())
print(ucdp_spans.shape)

ucdp_cy.to_csv(CLEAN + "ucdp_cy.csv", index=False)
ucdp_spans.to_csv(CLEAN + "ucdp_spans.csv", index=False)

**Checkpoint.** Run this cell. OK means done. Do not change it.

In [ ]:
ucdp_cy_check = pd.read_csv(CLEAN + "ucdp_cy.csv")
ucdp_spans_check = pd.read_csv(CLEAN + "ucdp_spans.csv")
assert list(ucdp_cy_check.columns) == ["iso3", "year", "n_conflicts", "max_intensity", "conflict_active"]
assert list(ucdp_spans_check.columns) == ["iso3", "start", "end", "max_intensity"]
assert ucdp_cy_check.duplicated(["iso3", "year"]).sum() == 0
assert ucdp_cy_check["iso3"].isna().sum() == 0
assert ucdp_cy_check["year"].min() >= 1950, "keep only UCDP years from 1950 on (sandbox B1 step 4)"
assert (ucdp_spans_check["end"] - ucdp_spans_check["start"] + 1).sum() == len(ucdp_cy_check)
keys = set(zip(ucdp_cy_check["iso3"], ucdp_cy_check["year"]))
n_starts = sum(1 for (cc, y) in keys if (cc, y - 1) not in keys)
assert len(ucdp_spans_check) == n_starts, "one span per uninterrupted run of conflict years"
print("OK")

## Decisions in this notebook

Write every choice you made here (and copy it into the decisions log in `CONVENTIONS.md`). You need these for the report.

| Decision | Why |
| --- | --- |
|  |  |